# 06.1 - RAG - Pipeline

This notebook builds a RAG pipeline over the medRxiv corpus indexed in notebook 07.2. For each query it:

1. **Retrieves** semantically relevant paper chunks from the Chroma vector database
2. **Formats** them as a numbered reference list
3. **Generates** a grounded answer with `[N]` citations using a local LLM via Ollama

The `output/medrxiv.db` database must exist before running this notebook (built by `scripts/07.2_medrxiv_knowledge_base.py`).

## 01. Configuration

Switch `model_id` to any LiteLLM-compatible model string — `"gpt-4o"`, `"anthropic/claude-sonnet-4-6"`, or another local Ollama model.

In [ ]:
import genscai.paths as paths

chroma_path = str(paths.output / "medrxiv.db")
collection_name = "articles_cosign_chunked_256"
model_id = "ollama/qwen2.5:7b"
n_results = 8  # chunks retrieved per query

## 02. Chroma Vector Database

Connect to the persistent Chroma database and verify the collection. The `articles_cosign_chunked_256` collection stores 256-character abstract chunks with 50-character overlap, indexed with cosine similarity.

In [ ]:
import chromadb

client = chromadb.PersistentClient(path=chroma_path)
collection = client.get_collection(collection_name)

print(f"Collection : {collection_name}")
print(f"Chunks     : {collection.count():,}")

## 03. Retrieval

`retrieve()` queries Chroma and returns a list of dicts, each containing the chunk text, cosine distance score, and article metadata (title, authors, date, link).

In [ ]:
def retrieve(query: str, n: int = n_results) -> list[dict]:
    results = collection.query(query_texts=[query], n_results=n)
    chunks = results["documents"][0]
    metas = results["metadatas"][0]
    scores = results["distances"][0]
    return [{"text": chunk, "score": score, **meta} for chunk, meta, score in zip(chunks, metas, scores)]

In [ ]:
# Inspect raw retrieval results for a sample query
hits = retrieve("SEIR model vaccination effectiveness")

for i, h in enumerate(hits[:3]):
    print(f"[{i + 1}] {h.get('title', 'N/A')}")
    print(f"    Date  : {str(h.get('date', ''))[:10]}")
    print(f"    Score : {h['score']:.4f}  (lower = more similar)")
    print(f"    Chunk : {h['text'][:180]}...")
    print()

## 04. Generation

Format the retrieved chunks as a numbered reference list, then pass them to the LLM with a system prompt that instructs it to cite sources and stay within the provided evidence.

In [ ]:
def format_context(hits: list[dict]) -> str:
    parts = []
    for i, h in enumerate(hits, 1):
        parts.append(
            f"[{i}] {h.get('title', 'Unknown title')} ({str(h.get('date', ''))[:10]})\n"
            f"    Authors : {h.get('authors', 'N/A')}\n"
            f"    Excerpt : {h['text']}\n"
            f"    Link    : {h.get('link', 'N/A')}"
        )
    return "\n\n".join(parts)

In [ ]:
from litellm import completion

SYSTEM_PROMPT = (
    "You are a research assistant specializing in infectious disease modeling. "
    "Answer the question using ONLY the provided research paper excerpts. "
    "Cite sources with [N] notation. "
    "If the excerpts do not contain enough information to answer the question, say so explicitly."
)


def generate(query: str, context: str) -> str:
    user_msg = f"Research paper excerpts:\n\n{context}\n\nQuestion: {query}"
    response = completion(
        model=model_id,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        temperature=0.3,
    )
    return response.choices[0].message.content

## 05. RAG Pipeline

Compose retrieval and generation into a single function and run a test query.

In [ ]:
def rag(query: str, n: int = n_results) -> dict:
    hits = retrieve(query, n=n)
    context = format_context(hits)
    answer = generate(query, context)
    return {"answer": answer, "sources": hits}

In [ ]:
result = rag("How are SEIR models used to evaluate the effectiveness of vaccination campaigns?")

print(result["answer"])
print("\n--- Sources ---")
for i, src in enumerate(result["sources"], 1):
    print(f"  [{i}] {src.get('title', 'N/A')}")
    print(f"       {src.get('link', '')}")

## 06. Research Queries

Run a set of representative infectious disease modeling questions through the RAG pipeline.

In [ ]:
queries = [
    "What methods are used to estimate the basic reproduction number R0?",
    "How do agent-based models compare to compartmental models for simulating disease spread?",
    "What are the main challenges in modeling COVID-19 transmission dynamics?",
    "How is spatial heterogeneity incorporated into infectious disease models?",
]

for q in queries:
    print(f"\nQ: {q}")
    result = rag(q)
    # Print first 600 chars of the answer
    print(f"A: {result['answer'][:600]}")
    if len(result["answer"]) > 600:
        print("   [...]")
    print(f"   ({len(result['sources'])} chunks retrieved)")
    print("-" * 60)